# Deposit-fee scenarios - Uhishange

## Current decision
ADR 0006 charges the supplier 10% of the retained gross deposit, with no minimum and independent of final service value. A EUR 100 deposit produces a EUR 10 fee and EUR 90 supplier allocation. Under the disclosed standard-card/VAT-inclusive scenario, contribution is EUR 6.31 and an assumed EUR 300/month budget needs 571 retained participations per year, before other variable costs. Actual costs and volumes have not been supplied.

Earlier 4% deposit and contract-price scenarios below are historical comparisons. Already frozen agreements retain their original terms. All application payments remain TEST.

## Context and methods

Decision: can the deposit fee fund ongoing operation? One unit is one association's retained deposit in one procurement, not a whole group or unique association. Fee rounded half-up to cents, matching `src/server/domain.ts`. Annual threshold is fixed annual costs divided by contribution, rounded up. All current pilot payments are TEST and generate no real income.

### Historical 4% baseline assumptions
- 4% deposit fee; deposit amounts EUR 50/100/200/500.
- Portal pays standard EEA card processing: 1.5% + EUR 0.25/capture.
- Alternate tax scenario: 4% fee includes 24% VAT; net fee rounded per participation.
- Monthly budgets EUR 50 (technical) and EUR 300 (technical 50 + four hours at 50 + admin/reserve 50). These are planning choices, not quotes.
- Base case excludes refunds, Connect/payout costs, variable support, acquisition, development recovery and profit.
- Separate 20% refund sensitivity assumes no platform fee on refunded captures and no returned capture fee. Expected volumes are not exact batch guarantees.
- No input is measured usage, a real invoice, or a market forecast. See PricingReview.md for the source/implementation audit and proposed next decision.


## Inputs and sources

Verified on 2026-10-10:
- [Historical 4% deposit-fee and ledger code](https://github.com/viljarpuusepp/uhishange/blob/685e70b323ab1c7d5cc221f2755fa26b25090340/src/server/domain.ts), plus `src/shared/constants.ts` and `src/server/validation.ts`.
- `docs/Uhishange_arianaluus_2026-10-09.docx`, sections 5.1/5.2/9.1/13.3: original fee is on full GMV, 500 × EUR 6,000 × 4% = EUR 120,000; deposits EUR 50–200 are a hypothesis.
- [Stripe Estonia pricing](https://stripe.com/en-ee/pricing): standard EEA 1.5% + EUR 0.25; premium EEA 2.8% + EUR 0.25. Standard-card modeling is optimistic for premium/business-card use.
- [Stripe Connect](https://stripe.com/en-ee/connect/pricing): additional fees can apply depending on integration; excluded below.
- [Stripe refunded fees](https://support.stripe.com/questions/understanding-fees-for-refunded-payments): original processing fees are not returned.
- [Estonian Tax and Customs Board](https://www.emta.ee/en/business-client/taxes-and-payment/value-added-tax): standard VAT 24%; no determination of the portal's registration status.
- [Hetzner reference prices](https://docs.hetzner.com/general/infrastructure-and-availability/price-adjustment/): CX33 EUR 8.49/month excluding IPv4; [backups](https://docs.hetzner.com/cloud/billing/faq/) 20% of server cost. Reference price, not an operational quote/capacity test.


In [1]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_CEILING
from html import escape
from IPython.display import display, HTML

FEE_BPS = 400
PSP_BPS = 150
PSP_FIXED_CENTS = 25
VAT_RATE = Decimal('0.24')
DEPOSITS_CENTS = [5000, 10000, 20000, 50000]
MONTHLY_BUDGETS_CENTS = [5000, 30000]

def basis_points(amount_cents, bps):
    return (amount_cents * bps + 5000) // 10000

def economics(deposit_cents, vat_included=False, psp_bps=PSP_BPS):
    fee = basis_points(deposit_cents, FEE_BPS)
    psp = basis_points(deposit_cents, psp_bps) + PSP_FIXED_CENTS
    net_fee = int((Decimal(fee) / (1 + VAT_RATE)).quantize(Decimal('1'), rounding=ROUND_HALF_UP)) if vat_included else fee
    return {'deposit': deposit_cents, 'fee': fee, 'psp': psp, 'net_fee': net_fee, 'contribution': net_fee - psp}

def break_even(monthly_cents, contribution_cents):
    return None if contribution_cents <= 0 else int((Decimal(monthly_cents * 12) / Decimal(contribution_cents)).to_integral_value(rounding=ROUND_CEILING))

def euro(cents):
    return f'EUR {Decimal(cents) / 100:,.2f}'

def table(headers, rows):
    head = ''.join(f'<th>{escape(str(value))}</th>' for value in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(value))}</td>' for value in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))


## Results

### Per retained participation
Contribution is available for fixed costs, not profit. The final column assumes VAT-inclusive fees and per-participation cent rounding.


In [2]:
rows = []
for deposit in DEPOSITS_CENTS:
    base, taxed = economics(deposit), economics(deposit, True)
    rows.append([euro(deposit), euro(base['fee']), euro(base['psp']), euro(base['contribution']), euro(taxed['contribution'])])
table(['Deposit', 'Gross fee', 'Capture cost', 'Contribution before VAT', 'Contribution with inclusive VAT'], rows)


Deposit,Gross fee,Capture cost,Contribution before VAT,Contribution with inclusive VAT
EUR 50.00,EUR 2.00,EUR 1.00,EUR 1.00,EUR 0.61
EUR 100.00,EUR 4.00,EUR 1.75,EUR 2.25,EUR 1.48
EUR 200.00,EUR 8.00,EUR 3.25,EUR 4.75,EUR 3.20
EUR 500.00,EUR 20.00,EUR 7.75,EUR 12.25,EUR 8.38


### Annual retained participations needed
One snow season can produce annual, rather than monthly, cash inflows. Additional refunds, supplier payout costs and variable support increase the required volume.


In [3]:
rows = []
for deposit in DEPOSITS_CENTS:
    base, taxed = economics(deposit), economics(deposit, True)
    rows.append([euro(deposit)] + [break_even(budget, case['contribution']) for budget in MONTHLY_BUDGETS_CENTS for case in (base, taxed)])
table(['Deposit', 'EUR 50/month before VAT', 'EUR 50/month inclusive VAT', 'EUR 300/month before VAT', 'EUR 300/month inclusive VAT'], rows)


Deposit,EUR 50/month before VAT,EUR 50/month inclusive VAT,EUR 300/month before VAT,EUR 300/month inclusive VAT
EUR 50.00,600,984,3600,5902
EUR 100.00,267,406,1600,2433
EUR 200.00,127,188,758,1125
EUR 500.00,49,72,294,430


### 500 retained participations at EUR 100
The original full-service-value scenario and the implemented deposit-fee scenario have different fee bases. Comparing the two does not establish actual adoption or revenue.


In [4]:
base, taxed = economics(10000), economics(10000, True)
table(['Metric', 'Annual value'], [
    ['Gross deposit fees: 500 × EUR 100 × 4%', euro(500 * base['fee'])],
    ['Capture costs', euro(500 * base['psp'])],
    ['Contribution before VAT', euro(500 * base['contribution'])],
    ['Contribution with inclusive VAT', euro(500 * taxed['contribution'])],
    ['Original full-GMV assumption: 500 × EUR 6,000 × 4%', euro(500 * basis_points(600000, FEE_BPS))],
])


Metric,Annual value
Gross deposit fees: 500 × EUR 100 × 4%,"EUR 2,000.00"
Capture costs,EUR 875.00
Contribution before VAT,"EUR 1,125.00"
Contribution with inclusive VAT,EUR 740.00
"Original full-GMV assumption: 500 × EUR 6,000 × 4%","EUR 120,000.00"


### Sensitivities and calculation checks
Refund rate is a scenario. Expected capture costs per retained success equal capture cost divided by the retained fraction. Premium cards and five minutes of variable support at EUR 50/hour illustrate why the base table is optimistic.


In [5]:
retained_fraction = Decimal('0.8')
expected_base = Decimal(base['net_fee']) - Decimal(base['psp']) / retained_fraction
expected_taxed = Decimal(taxed['net_fee']) - Decimal(taxed['psp']) / retained_fraction
table(['EUR 100 deposit sensitivity', 'Contribution per retained success', 'Annual successes for EUR 300/month'], [
    ['20% refunded captures, before VAT', euro(expected_base), break_even(30000, expected_base)],
    ['20% refunded captures, inclusive VAT', euro(expected_taxed), break_even(30000, expected_taxed)],
    ['Premium EEA card, inclusive VAT', euro(economics(10000, True, 280)['contribution']), break_even(30000, economics(10000, True, 280)['contribution'])],
    ['5 minutes support at assumed EUR 50/hour, before VAT', euro(Decimal(base['contribution']) - Decimal(5000) * 5 / 60), 'No positive contribution'],
])

# Independent Decimal calculation validates the integer fee rule.
for amount in [1, 12, 13, 100, 5000, 10000, 10003, 20000, 50000, 100000000]:
    independently_rounded = int((Decimal(amount) * Decimal('0.04')).quantize(Decimal('1'), rounding=ROUND_HALF_UP))
    assert basis_points(amount, FEE_BPS) == independently_rounded
    assert 0 <= basis_points(amount, FEE_BPS) <= amount
assert basis_points(12, FEE_BPS) == 0
assert basis_points(13, FEE_BPS) == 1
assert base['contribution'] == 225 and taxed['contribution'] == 148
assert break_even(30000, 225) == 1600
assert break_even(30000, 148) == 2433
assert break_even(30000, 0) is None
for deposit in DEPOSITS_CENTS:
    for vat_included in (False, True):
        result = economics(deposit, vat_included)
        assert result['fee'] == result['net_fee'] + (result['fee'] - result['net_fee'])
        for budget in MONTHLY_BUDGETS_CENTS:
            count = break_even(budget, result['contribution'])
            assert count * result['contribution'] >= budget * 12
            assert (count - 1) * result['contribution'] < budget * 12
print('All fee, VAT-scenario, boundary and break-even assertions passed.')


EUR 100 deposit sensitivity,Contribution per retained success,Annual successes for EUR 300/month
"20% refunded captures, before VAT",EUR 1.81,1987
"20% refunded captures, inclusive VAT",EUR 1.04,3454
"Premium EEA card, inclusive VAT",EUR 0.18,20000
"5 minutes support at assumed EUR 50/hour, before VAT",EUR -1.92,No positive contribution


All fee, VAT-scenario, boundary and break-even assertions passed.


## Historical contract-price tariff comparison
ADR 0005 chose 4% of final gross contract value with a EUR 10 minimum. This policy is superseded for new agreements by ADR 0006, but remains binding for already frozen contracts. The following historical scenarios assume deposit coverage, standard EEA cards, fee-inclusive VAT and no refunds/other variable costs. They do not measure actual revenue or customer willingness.

In [6]:
current_rows = []
for price_cents, deposit_cents in [(19000, 3000), (31000, 7500), (70000, 10000), (600000, 30000)]:
    fee_cents = max(1000, basis_points(price_cents, FEE_BPS))
    assert fee_cents <= deposit_cents
    net_fee_cents = int((Decimal(fee_cents) / (1 + VAT_RATE)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))
    capture_cents = economics(deposit_cents)['psp']
    contribution_cents = net_fee_cents - capture_cents
    current_rows.append([euro(price_cents), euro(deposit_cents), euro(fee_cents), euro(deposit_cents - fee_cents), euro(contribution_cents), break_even(30000, contribution_cents)])
table(['Final contract price', 'Deposit', 'Historical gross fee', 'Supplier deposit payout', 'Contribution, inclusive VAT', 'Annual contracts for EUR 300/month'], current_rows)
assert current_rows[2][4] == 'EUR 20.83' and current_rows[2][5] == 173
print('Historical contract-price assertions passed.')


Final contract price,Deposit,Historical gross fee,Supplier deposit payout,"Contribution, inclusive VAT",Annual contracts for EUR 300/month
EUR 190.00,EUR 30.00,EUR 10.00,EUR 20.00,EUR 7.36,490
EUR 310.00,EUR 75.00,EUR 12.40,EUR 62.60,EUR 8.62,418
EUR 700.00,EUR 100.00,EUR 28.00,EUR 72.00,EUR 20.83,173
"EUR 6,000.00",EUR 300.00,EUR 240.00,EUR 60.00,EUR 188.80,20


Historical contract-price assertions passed.


## Current 10% deposit tariff
Same disclosed cost assumptions, no refunds or extra variable costs. Changing service price or the achieved tier does not change the deposit fee.

In [7]:
rows = []
for deposit in DEPOSITS_CENTS:
    fee = basis_points(deposit, 1000)
    net_fee = int((Decimal(fee) / (1 + VAT_RATE)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))
    contribution = net_fee - economics(deposit)['psp']
    rows.append([euro(deposit), euro(fee), euro(deposit - fee), euro(contribution), break_even(30000, contribution)])
table(['Deposit', 'Gross 10% fee', 'Supplier allocation', 'Contribution, inclusive VAT', 'Annual retained participations for EUR 300/month'], rows)
assert rows[1][1:4] == ['EUR 10.00', 'EUR 90.00', 'EUR 6.31']
assert rows[1][4] == 571
assert basis_points(3000, 1000) == 300  # no EUR 10 minimum
print('Current deposit-tariff assertions passed.')

Deposit,Gross 10% fee,Supplier allocation,"Contribution, inclusive VAT",Annual retained participations for EUR 300/month
EUR 50.00,EUR 5.00,EUR 45.00,EUR 3.03,1189
EUR 100.00,EUR 10.00,EUR 90.00,EUR 6.31,571
EUR 200.00,EUR 20.00,EUR 180.00,EUR 12.88,280
EUR 500.00,EUR 50.00,EUR 450.00,EUR 32.57,111


Current deposit-tariff assertions passed.


## Takeaways
The current 10% deposit tariff is predictable before final service value is known. Profitability remains volume- and cost-dependent. Historical policies remain frozen. No application data is changed by this notebook.